In [1]:
import torch
import torch.nn as nn
import re
import numpy as np
from torch.utils.data import DataLoader, Dataset
import difflib
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import torch.nn.functional as F

# Hyperparameters definition

In [3]:
# It is arbitrary values
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
N_LAYERS = 2
DROPOUT = 0.5
N_EPOCHS = 100
LR = 3e-3
BATCH_SIZE = 32
SEQ_LEN = 30

# Input preprocessing

In [18]:
def replace_contractions(text):
    """Replace every common contractions by their complete expression in the given text"""
    contractions_dict = {
        "he's": "he is",
        "i'm": "I am",
        "you're": "you are",
        "we've": "we have",
        "they've": "they have",
        "don't": "do not",
        "isn't": "is not",
        "it's": "it is",
        "didn't": "did not",
        "aren't": "are not",
        "let's": "let us",
        "couldn't": "could not",
        "wasn't": "was not",
        "weren't": "were not",
        "ain't": "am not",
        "i've": "I have",
        "that's": "that is",
        "i'll": "I will",
        "you'd": "you would",
        "they're": "they are",
        "i won't": "I will not",
        "can't": "cannot",
        "you've": "you have",
        "there's": "there is",
        "won't": "will not",
        "you'll": "you will",
        "doesn't": "does not",
        "must've": "must have",
        "what's": "what is",
        "we're": "we are",
        "haven't": "have not",
        "wouldn't": "would not",
        "i'd": "I would",
        "she's": "she is",
        "nobody's": "nobody is",
        "we'll": "we will",
        "they'd": "they would",
        "mustn't": "must not",
        "could've": "could have",
        "shouldn't": "should not",
        "he'll": "he will",
        "he'd": "he would",
        "hadn't": "had not",
        "where'd": "where did",
        "we'd": "we would",
    }

    pattern = re.compile(r'\b(' + '|'.join(re.escape(key) for key in contractions_dict.keys()) + r')\b')
    
    return pattern.sub(lambda x: contractions_dict[x.group()], text)

In [29]:
def preprocess_text(input_filename = "adele.txt"):
    with open(input_filename, "r") as f:
        original_sentences = f.readlines()

    processed_sentences = []
    for sentence in original_sentences:
        sentence_contraction = sentence.lower()
        sentence_contraction = replace_contractions(sentence_contraction)
        sentence_contraction = sentence_contraction.replace("\n", " <EOS>")
        
        # Remove all non-alphanumeric character to only conserve tokens
        sentence_contraction = re.sub(r'[^a-zA-Z\s\<\>]', '', sentence_contraction)
        
        # Split strings to obtain the list of tokens 
        sentence_contraction = sentence_contraction.split()
        if len(sentence_contraction) > 0:
            processed_sentences.append(sentence_contraction)

    # Create word-to-index and index-to-word mappings for the text encoding
    word2idx = {word: idx for idx, word in enumerate(set(word for sentence in processed_sentences for word in sentence))}
    idx2word = {idx: word for word, idx in word2idx.items()}
    
    split_index = int(len(original_sentences) * 0.8)

    train_sentences = processed_sentences[:split_index]
    test_sentences_original = processed_sentences[split_index:]
    
    test_sentences_processed = []
    for sentence in test_sentences_original:
        # Remove test samples with unknown words
        has_unknown_words = False
        for word in sentence:
            if word2idx.get(word) == None:
                has_unknown_words = True
                break
        # Remove test samples with a size less than 3 to enable to evaluate these sentences
        if not has_unknown_words and len(sentence) >= 3:
            test_sentences_processed.append(sentence)

    print(f"Number of training sentences: {len(train_sentences)}")
    print(f"Number of validation sentences: {len(test_sentences_processed)}")
    vocab_size = len(word2idx)
    print(f"Vocabulary size: {vocab_size}")

    return train_sentences, test_sentences_processed, word2idx, idx2word, vocab_size

In [5]:
class IndexedTextDataset(Dataset):
    def __init__(self, sentences, word2idx, max_len=None):
        """ Dataset class that returns word indices for sentences.

        Args:
        - sentences (list[list[str]]): The sentences, each represented as a list of words.
        - word2idx (dict): Mapping from words to indices.
        - max_len (int, optional): Maximum sentence length. Sentences longer than this are truncated.
        """
        self.sentences = sentences
        self.word2idx = word2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        
        indexed_sentence = [self.word2idx.get(word) for word in sentence]  # Use None for unknown words
        
        if self.max_len:
            indexed_sentence = indexed_sentence[:self.max_len]
        
        X = indexed_sentence[:-1]
        y = indexed_sentence[1:]
        
        return X, y

In [6]:
def convert_batch(batch):
    """ Pad the variating length data and target to be able to convert the data into a tensor. \
        Note that the X and y vector must have the same size

    Args:
        batch (List[tuple[list, list]]): data to pad. Should be generated from the `getitem` function of `IndexedTextDataset`
    Returns:
        (tensor, tensor, tensor): X_padded, y_padded and the original size of the X and y vectors
    """
    X_batch, y_batch = zip(*batch)
    
    X_padded = pad_sequence([torch.tensor(x) for x in X_batch], batch_first=True, padding_value=0)  # Use padding_value=0
    y_padded = pad_sequence([torch.tensor(y) for y in y_batch], batch_first=True, padding_value=0)
    
    lengths = torch.tensor([len(x) for x in X_batch])

    return X_padded, y_padded, lengths

In [30]:
train_sentences, test_sentences, word2idx, idx2word, vocab_size = preprocess_text()

train_dataset = IndexedTextDataset(train_sentences, word2idx)
val_dataset = IndexedTextDataset(test_sentences, word2idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=convert_batch)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=convert_batch)

for X, y, X_lengths in train_loader:
    print("Example of X shape for train loader:", X.shape)
    print("Example of y shape from train loader:", y.shape)
    print("Example of X lengths from train loader:", X_lengths)
    break

Number of training sentences: 1920
Number of validation sentences: 476
Vocabulary size: 1344
Example of X shape for train loader: torch.Size([32, 22])
Example of y shape from train loader: torch.Size([32, 22])
Example of X lengths from train loader: tensor([ 5, 11,  9,  7, 10, 12,  5,  5,  9,  6, 12,  5, 15, 12,  6,  9,  6,  2,
        14, 15,  5, 18,  5,  8, 14,  9,  7,  6, 22,  8,  9,  5])


# Model defintion

In [ ]:
class NLP(nn.Module):
    def __init__(self, vocab_size, hidden_dim, n_layers, dropout, model_type='LSTM'):
        super(NLP, self).__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.num_layers = n_layers
        self.rnn_type = model_type
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # The embedding matrix for one hot encoding is a identity matrix of size vocab_size
        self.embedding = nn.Embedding.from_pretrained(torch.eye(vocab_size, dtype=torch.float))

        if model_type == 'LSTM':
            self.nlp = nn.LSTM(vocab_size, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        elif model_type == 'GRU':
            self.nlp = nn.GRU(vocab_size, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        else:
            raise Exception("Model type not supported")
        
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, lengths, hidden):
        x = self.embedding(x)
        
        # Pack padded sequences for efficient processing
        packed_input = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        
        packed_output, hidden = self.nlp(packed_input, hidden)

        # Unpack the sequence to be able to provide it to the FC layer
        output, _ = pad_packed_sequence(packed_output, batch_first=True)

        x = self.fc(output)
        return x, hidden
    
    def init_hidden(self, batch_size):
        if self.rnn_type == 'LSTM':
            # LSTM requires both hidden state and cell state
            hidden = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(self.device)
            cell = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(self.device)
            return (hidden, cell)
        else:
            # GRU only requires the hidden state
            hidden = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(self.device)
            return hidden

In [ ]:
def train(model, dataloader, n_epochs, lr, filename):
    """ Train the given model for n_epoch.

    Args:
        model (nn.Module): The instance of the model to train
        dataloader (Dataloader): dataloader that provide the training data
        n_epochs (int): the number of epoch to perform
        lr (float): learning rate used in the optimizer
        filename (str): name of the file used to save the final model
    """

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device = {device}")
    torch.cuda.empty_cache()
    model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(reduction='mean')

    model.train()

    for epoch in range(n_epochs):
        train_losses = []
        for i, (X, y, lengths) in enumerate(dataloader):
            
            hidden = model.init_hidden(X.shape[0])
            X, y, lengths = X.to(device), y.to(device), lengths.to(device)

            optimizer.zero_grad()
            output, hidden = model(X, lengths, hidden)

            # Resize vectors to a shape accepted by the loss function
            output = output.view(-1, vocab_size)
            y_padded = y.view(-1)
            
            loss = criterion(output, y_padded.to(model.device))
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            if i % 10 == 0:
                print(f"Epoch {epoch}, step {i}, loss {loss.item()}")
        
        print(f"Epoch {epoch} finished. Train loss: {np.array(train_losses).mean()}, Perplexity: {np.exp(np.array(train_losses).mean())}")

    torch.save(model.state_dict(), f"model_save/{filename}.pth")

In [10]:
model_type = 'GRU'
GRU_model = NLP(vocab_size, HIDDEN_DIM, N_LAYERS, DROPOUT, model_type)

train(GRU_model, train_loader, N_EPOCHS, LR, f"{model_type}_model_OneHot_100")

device = cpu
Epoch 0, step 0, loss 7.218798637390137
Epoch 0, step 10, loss 6.356459617614746
Epoch 0, step 20, loss 6.350924491882324
Epoch 0, step 30, loss 6.304145336151123
Epoch 0, step 40, loss 6.114578723907471
Epoch 0, step 50, loss 6.222109794616699
Epoch 0 finished. Train loss: 6.217017388343811, Perplexity: 501.20609729642274
Epoch 1, step 0, loss 6.081414699554443
Epoch 1, step 10, loss 6.036034107208252
Epoch 1, step 20, loss 5.513484477996826
Epoch 1, step 30, loss 5.829812049865723
Epoch 1, step 40, loss 5.89060115814209
Epoch 1, step 50, loss 5.425445079803467
Epoch 1 finished. Train loss: 5.790414611498515, Perplexity: 327.14863584713646
Epoch 2, step 0, loss 5.648214340209961
Epoch 2, step 10, loss 5.599354267120361
Epoch 2, step 20, loss 5.8705949783325195
Epoch 2, step 30, loss 5.4833807945251465
Epoch 2, step 40, loss 5.725640296936035
Epoch 2, step 50, loss 5.4333672523498535
Epoch 2 finished. Train loss: 5.496774951616923, Perplexity: 243.90406008954523
Epoch 3, s

In [11]:
model_type = 'LSTM'
LSTM_model = NLP(vocab_size, HIDDEN_DIM, N_LAYERS, DROPOUT, model_type)

train(LSTM_model, train_loader, N_EPOCHS, LR, f"{model_type}_model_OneHot_100")

device = cpu
Epoch 0, step 0, loss 7.24015474319458
Epoch 0, step 10, loss 6.258553504943848
Epoch 0, step 20, loss 6.332048416137695
Epoch 0, step 30, loss 6.048920631408691
Epoch 0, step 40, loss 5.986612319946289
Epoch 0, step 50, loss 6.010255336761475
Epoch 0 finished. Train loss: 6.242411589622497, Perplexity: 514.0968077373409
Epoch 1, step 0, loss 5.731386184692383
Epoch 1, step 10, loss 5.936934471130371
Epoch 1, step 20, loss 5.877802848815918
Epoch 1, step 30, loss 5.761813640594482
Epoch 1, step 40, loss 5.907402515411377
Epoch 1, step 50, loss 5.732020378112793
Epoch 1 finished. Train loss: 5.855617014567057, Perplexity: 349.19028909311015
Epoch 2, step 0, loss 5.453052520751953
Epoch 2, step 10, loss 5.588271141052246
Epoch 2, step 20, loss 5.717025279998779
Epoch 2, step 30, loss 5.646935939788818
Epoch 2, step 40, loss 5.440098762512207
Epoch 2, step 50, loss 5.559040069580078
Epoch 2 finished. Train loss: 5.6011933326721195, Perplexity: 270.749308719551
Epoch 3, step 0

# Evaluation of the model

In [12]:
GRU_model = NLP(vocab_size, HIDDEN_DIM, N_LAYERS, DROPOUT, 'GRU')
GRU_model.load_state_dict(torch.load("model_save/GRU_model_OneHot_100.pth", weights_only=True))

LSTM_model = NLP(vocab_size, HIDDEN_DIM, N_LAYERS, DROPOUT, 'LSTM')
LSTM_model.load_state_dict(torch.load("model_save/LSTM_model_OneHot_100.pth", weights_only=True))

<All keys matched successfully>

In [13]:
def computer_word_similarity(reference, prediction):
    """
    Function to compute the similarity between the reference text and the
    predicted text (similarity between the 2 sequences at the word level).

    Args:
        reference: Reference text
        prediction: Predicted text

    Returns:
        Similarity ratio between words
    """
    ref_words = reference.split()
    pred_words = prediction.split()

    matcher = difflib.SequenceMatcher(None, ref_words, pred_words)

    ratio = matcher.ratio()

    return ratio

In [ ]:
def generate_text_with_metrics(model, word2idx, idx2word, start_word, full_text, num_words=10, random_sample=False, top_k=10):
    """
    Generate text based on the trained model output and compute evaluation metrics.
    
    Args:
        model: The trained PyTorch model.
        word2idx: Dictionary mapping words to their indices.
        idx2word: Dictionary mapping indices to their words.
        start_word: The initial word to start generating text.
        full_text: The full original text (start words + words to predict).
        num_words: Number of words to generate.
        random_sample: If True, generated words will be sampled from the distribution instead of taking the one with greatest probability.
        top_k: Top-k accuracy to compute.
    
    Returns:
        metrics: A dictionary containing accuracy, top-k accuracy, perplexity and the similarity ratio.
        generated_text: The generated sequence of words.
    """
    model.eval()
    
    start_word = start_word.split()
    start_word_size = len(start_word)
    
    # Initialize the generated word with the first word to be able to better assess the quality of the generation
    generated_words = start_word
    
    input_tensor = torch.tensor([[word2idx[word] for word in start_word]], dtype=torch.long).to(model.device)
    
    hidden = model.init_hidden(batch_size=1)
    log_probs = []
    predictions = []
    top_k_acc = 0
    
    for i in range(num_words):
        with torch.no_grad():
            output, hidden = model(input_tensor, torch.tensor([1]), hidden)
            
            # Remove batch dimension: [seq_len, vocab_size]
            output = output.squeeze(0)  
            
            # Get probabilities for last time step
            probabilities = F.softmax(output[-1], dim=0).cpu().numpy()  

            top_k_indices = np.argsort(-probabilities)[:top_k][::-1]
            
            for idx in top_k_indices:
                top_k_next_word = idx2word[idx]
                if i + start_word_size < len(full_text):
                    target_word = full_text[i + start_word_size]
                    if top_k_next_word == target_word:
                        top_k_acc += 1
                        break
        
        # The random sampling is done after the top_k accuracy evaluation as it's shouldn't influancate the metrics computation
        if random_sample:
            generated_idx = np.random.choice(len(probabilities), p=probabilities)
        else:
            generated_idx = np.argmax(probabilities)

        next_word = idx2word[generated_idx]

        # Stop generation if end-of-sequence token is reached
        if next_word == "<EOS>":
            break

        generated_words.append(next_word)
        predictions.append(next_word)
        log_probs.append(np.log(probabilities[generated_idx]))

        input_tensor = torch.tensor([[generated_idx]], dtype=torch.long).to(model.device)

    # Isolate the word that must have been predicted
    targets = full_text[start_word_size:]
    accuracy = 0
    for i in range(len(predictions)):
        if i < len(targets) and predictions[i] == targets[i]:
            accuracy += 1
    
    accuracy /= len(targets)
    
    top_k_acc /= len(targets)
    
    perplexity = np.exp(-np.mean(log_probs)) if log_probs else float('inf')
    
    metrics = {
        "accuracy": accuracy,
        "top_k_accuracy": top_k_acc,
        "perplexity": perplexity,
        'similarity ratio': computer_word_similarity(' '.join(targets), ' '.join(predictions))
    }
    
    generated_text = ' '.join(generated_words)
    return metrics, generated_text


# Quantitative analysis of the models on the test set

In [15]:
def evaluate(test_sentences, model, word2idx, idx2word):
    # Compute the metrics for all testing sequences
    results = []
    for sentences in test_sentences:
        metrics, _ = generate_text_with_metrics(model, word2idx, idx2word, " ".join(sentences[0:2]), sentences ,num_words=len(sentences)-2, random_sample=False)
        results.append(metrics)

    # Aggreate results into a single dictonary
    res = {}
    for key in results[0].keys():
        acc = []
        for result in results:
            if result[key] == float('inf'):
                continue
            acc.append(result[key])
        res[key] = np.array(acc).mean()

    return res
print(f"Metrics for the GRU Model: {evaluate(test_sentences, GRU_model,word2idx, idx2word)}")
print(f"Metrics for the LSTM Model: {evaluate(test_sentences, LSTM_model, word2idx, idx2word)}")

Metrics for the GRU Model: {'accuracy': 0.008140044793995075, 'top_k_accuracy': 0.12665645502527118, 'perplexity': 1.6294378, 'similarity ratio': 0.05808079785208645}
Metrics for the LSTM Model: {'accuracy': 0.006273532371858731, 'top_k_accuracy': 0.1329853330960887, 'perplexity': 1.7410561, 'similarity ratio': 0.05348279403538883}


# Qualitative analysis of the model on the test set

## GRU model:

In [ ]:
metrics, generated_text = generate_text_with_metrics(GRU_model, word2idx, idx2word, ' '.join(test_sentences[2][0:2]), test_sentences[2], num_words=len(test_sentences[2]) - 2, random_sample=False)
print(f"Target: {' '.join(test_sentences[2][:-2])}")
print(f"Generated text: {generated_text}")
print(f"Metrics values of the generated text: {metrics}", end="\n\n")

metrics, generated_text = generate_text_with_metrics(GRU_model, word2idx, idx2word, ' '.join(test_sentences[10][0:2]), test_sentences[10], num_words=len(test_sentences[10]) - 2, random_sample=False)
print(f"Target: {' '.join(test_sentences[10][:-2])}")
print(f"Generated text: {generated_text}")
print(f"Metrics values of the generated text: {metrics}", end="\n\n")

metrics, generated_text = generate_text_with_metrics(GRU_model, word2idx, idx2word, ' '.join(test_sentences[20][0:2]), test_sentences[20], num_words=len(test_sentences[20]) - 2, random_sample=False)
print(f"Target: {' '.join(test_sentences[20][:-2])}")
print(f"Generated text: {generated_text}")
print(f"Metrics values of the generated text: {metrics}", end="\n\n")

Target: to persist with this heartbreak and running around
Generated text (with no random_sampling): to persist tell you I am sorry
Metrics values of the generated text: {'accuracy': 0.0, 'top_k_accuracy': 0.0, 'perplexity': 1.3097576, 'similarity ratio': 0.0}

Target: but my knees were far too weak
Generated text: but my you will not
Metrics values of the generated text: {'accuracy': 0.0, 'top_k_accuracy': 0.0, 'perplexity': 2.2847886, 'similarity ratio': 0.0}

Target: cause i heard it screaming out your name
Generated text: cause i i knew that that was the last time
Metrics values of the generated text: {'accuracy': 0.0, 'top_k_accuracy': 0.0, 'perplexity': 1.3307492, 'similarity ratio': 0.0}



## LSTM model

In [17]:
metrics, generated_text = generate_text_with_metrics(LSTM_model, word2idx, idx2word, ' '.join(test_sentences[2][0:2]), test_sentences[2], num_words=len(test_sentences[2]) - 2, random_sample=False)
print(f"Target: {' '.join(test_sentences[2][:-2])}")
print(f"Generated text: {generated_text}")
print(f"Metrics values of the generated text: {metrics}", end="\n\n")

metrics, generated_text = generate_text_with_metrics(LSTM_model,word2idx, idx2word, ' '.join(test_sentences[10][0:2]), test_sentences[10], num_words=len(test_sentences[10]) - 2, random_sample=False)
print(f"Target: {' '.join(test_sentences[10][:-2])}")
print(f"Generated text: {generated_text}")
print(f"Metrics values of the generated text: {metrics}", end="\n\n")

metrics, generated_text = generate_text_with_metrics(LSTM_model, word2idx, idx2word, ' '.join(test_sentences[20][0:2]), test_sentences[20], num_words=len(test_sentences[20]) - 2, random_sample=False)
print(f"Target: {' '.join(test_sentences[20][:-2])}")
print(f"Generated text: {generated_text}")
print(f"Metrics values of the generated text: {metrics}", end="\n\n")

Target: to persist with this heartbreak and running around
Generated text: to persist make you feel my love i know you
Metrics values of the generated text: {'accuracy': 0.0, 'top_k_accuracy': 0.125, 'perplexity': 1.5180686, 'similarity ratio': 0.0}

Target: but my knees were far too weak
Generated text: but my i could not stay away i could
Metrics values of the generated text: {'accuracy': 0.0, 'top_k_accuracy': 0.14285714285714285, 'perplexity': 1.5358601, 'similarity ratio': 0.0}

Target: cause i heard it screaming out your name
Generated text: cause i i heard it screaming out your name your
Metrics values of the generated text: {'accuracy': 0.0, 'top_k_accuracy': 0.125, 'perplexity': 1.1913801, 'similarity ratio': 0.75}

